# Rates and Returns in Python
### CFA Level 1 Quantitative Methods | Implemented in Python with Real Market Data

---

## What this notebook covers

Before any statistical analysis, risk model, or portfolio optimisation,
you need to understand how returns are calculated.

The CFA curriculum distinguishes between several types of returns
that are not interchangeable — and confusing them is a common error
in both exams and professional practice.

**By the end of this notebook you will be able to:**
- Calculate holding period, simple, and log returns from raw price data
- Compare arithmetic and geometric mean returns and know when to use each
- Calculate money-weighted and time-weighted returns from cash flow data
- Annualise returns across different compounding frequencies
- Adjust nominal returns for inflation using real CPI data from FRED
- Calculate leveraged returns and understand their risk implications

**Data sources:**
- Price data — Financial Modeling Prep (FMP) API
- Inflation data — Federal Reserve Economic Data (FRED) API

**Prerequisites:** None — this is the first quantitative notebook

## Setup

Libraries we will use:

- `requests` — to pull data from FMP and FRED APIs
- `pandas` — for data manipulation
- `numpy` — for mathematical operations
- `matplotlib` — for visualizations
- `scipy` — for IRR calculation (MWRR)

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.optimize as optimize
import warnings
from dotenv import load_dotenv
import os

load_dotenv()
warnings.filterwarnings('ignore')

FMP_KEY  = os.getenv("FMP_KEY")
FRED_KEY = os.getenv("FRED_KEY")

# Plot styling
plt.rcParams['figure.facecolor'] = '#0D1117'
plt.rcParams['axes.facecolor']   = '#161B22'
plt.rcParams['axes.edgecolor']   = '#30363D'
plt.rcParams['text.color']       = '#E6EDF3'
plt.rcParams['axes.labelcolor']  = '#E6EDF3'
plt.rcParams['xtick.color']      = '#8B949E'
plt.rcParams['ytick.color']      = '#8B949E'
plt.rcParams['grid.color']       = '#21262D'
plt.rcParams['grid.linestyle']   = '--'
plt.rcParams['grid.alpha']       = 0.5

print("Libraries loaded successfully")
print("FMP Key loaded  ✓" if FMP_KEY  else "FMP Key missing ✗")
print("FRED Key loaded ✓" if FRED_KEY else "FRED Key missing ✗")

Libraries loaded successfully
FMP Key loaded  ✓
FRED Key loaded ✓


## 1. Holding Period Return

The **Holding Period Return (HPR)** is the most basic return measure.
It answers: *how much did I make or lose over a specific period?*

$$HPR = \frac{P_t - P_0 + D_t}{P_0}$$

Where:
- $P_t$ = ending price
- $P_0$ = beginning price  
- $D_t$ = dividends or cash flows received during the period

For price-only returns (no dividends):

$$HPR = \frac{P_t - P_0}{P_0} = \frac{P_t}{P_0} - 1$$

We pull **5 years of SPY prices** from FMP and compute HPR
over multiple horizons — daily, monthly, and the full period.

In [2]:
# =============================================================================
# CELL 5 — Pull Price Data & Compute Holding Period Returns
# =============================================================================

def get_prices(ticker, api_key, from_date="2019-01-01", to_date="2024-01-01"):
    """Pull historical daily closing prices from FMP."""
    url = "https://financialmodelingprep.com/stable/historical-price-eod/full"
    params = {"symbol": ticker, "from": from_date, "to": to_date, "apikey": api_key}
    response = requests.get(url, params=params)
    data = response.json()

    if isinstance(data, list) and len(data) > 0:
        df = pd.DataFrame(data)
    elif isinstance(data, dict) and "historical" in data:
        df = pd.DataFrame(data["historical"])
    else:
        print(f"Error fetching {ticker}: {str(data)[:200]}")
        return None

    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date").sort_index()
    return df["close"]

# Pull SPY prices
print("Pulling SPY price data from FMP...")
spy = get_prices("SPY", FMP_KEY)
print(f"  ✓ SPY — {len(spy)} days loaded")
print(f"  Period: {spy.index[0].date()} → {spy.index[-1].date()}")

# ── HOLDING PERIOD RETURNS ────────────────────────────────────────────────────

# Daily HPR
hpr_daily = (spy / spy.shift(1)) - 1

# Monthly HPR — resample to month end
spy_monthly = spy.resample("ME").last()
hpr_monthly = (spy_monthly / spy_monthly.shift(1)) - 1

# Full period HPR — from first to last price
P0       = spy.iloc[0]
Pt       = spy.iloc[-1]
hpr_full = (Pt - P0) / P0

# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("HOLDING PERIOD RETURNS — SPY (2019–2023)")
print(f"{'='*55}")
print(f"\n  Start Price (Jan 2019):  ${P0:.2f}")
print(f"  End Price   (Dec 2023):  ${Pt:.2f}")
print(f"\n  Full Period HPR:         {hpr_full:.2%}")
print(f"\n  Daily HPR — Summary:")
print(f"    Mean:   {hpr_daily.mean():.4%}")
print(f"    Best:   {hpr_daily.max():.4%}  ({hpr_daily.idxmax().date()})")
print(f"    Worst:  {hpr_daily.min():.4%}  ({hpr_daily.idxmin().date()})")
print(f"\n  Monthly HPR — Summary:")
print(f"    Mean:   {hpr_monthly.mean():.4%}")
print(f"    Best:   {hpr_monthly.max():.4%}  ({hpr_monthly.idxmax().date()})")
print(f"    Worst:  {hpr_monthly.min():.4%}  ({hpr_monthly.idxmin().date()})")
print(f"{'='*55}")
print(f"\nCFA Note: HPR makes no assumption about compounding.")
print(f"It is the raw return over whatever period you choose.")

Pulling SPY price data from FMP...
  ✓ SPY — 1258 days loaded
  Period: 2019-01-02 → 2023-12-29

HOLDING PERIOD RETURNS — SPY (2019–2023)

  Start Price (Jan 2019):  $250.18
  End Price   (Dec 2023):  $475.31

  Full Period HPR:         89.99%

  Daily HPR — Summary:
    Mean:   0.0599%
    Best:   9.0603%  (2020-03-24)
    Worst:  -10.9424%  (2020-03-16)

  Monthly HPR — Summary:
    Mean:   1.1051%
    Best:   12.6984%  (2020-04-30)
    Worst:  -12.9987%  (2020-03-31)

CFA Note: HPR makes no assumption about compounding.
It is the raw return over whatever period you choose.


## 2. Arithmetic vs Geometric Mean Return

The arithmetic mean return is the simple average of periodic returns.
The geometric mean return accounts for compounding.

$$\bar{r}_{arithmetic} = \frac{1}{T}\sum_{t=1}^{T} r_t$$

$$\bar{r}_{geometric} = \left(\prod_{t=1}^{T} (1 + r_t)\right)^{1/T} - 1$$

**When to use which — the CFA rule:**

| Use Case | Measure |
|----------|---------|
| Expected return for a single future period | Arithmetic mean |
| Actual compound growth over multiple periods | Geometric mean |
| Performance reporting | Geometric mean |
| Forward-looking portfolio optimization | Arithmetic mean |

The geometric mean is always **less than or equal to** the arithmetic mean.
The gap between them grows with volatility.

This is not a technicality — it has real consequences for investors
who confuse the two when evaluating manager performance.

In [3]:
# =============================================================================
# CELL 7 — Arithmetic vs Geometric Mean Return
# =============================================================================

r = hpr_daily.dropna().values
T = len(r)

# ── ARITHMETIC MEAN ───────────────────────────────────────────────────────────
arith_mean_daily = np.mean(r)
arith_mean_ann   = arith_mean_daily * 252

# ── GEOMETRIC MEAN ────────────────────────────────────────────────────────────
# Chain-link: multiply all (1 + r_t) then take the T-th root
geo_mean_daily = np.prod(1 + r) ** (1 / T) - 1
geo_mean_ann   = (1 + geo_mean_daily) ** 252 - 1

# Verify geometric mean against full period HPR
# (1 + geo_mean_daily)^T - 1 should equal full period HPR
verify = (1 + geo_mean_daily) ** T - 1

# ── THE GAP ───────────────────────────────────────────────────────────────────
# Approximation: geo ≈ arith - (variance / 2)
variance_daily = np.var(r, ddof=1)
gap_approx     = variance_daily / 2

# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
print(f"{'='*58}")
print("ARITHMETIC vs GEOMETRIC MEAN — SPY Daily Returns (2019–2023)")
print(f"{'='*58}")

print(f"\n{'─'*58}")
print("  DAILY")
print(f"{'─'*58}")
print(f"  Arithmetic Mean:    {arith_mean_daily:>10.6f}  ({arith_mean_daily:.4%})")
print(f"  Geometric Mean:     {geo_mean_daily:>10.6f}  ({geo_mean_daily:.4%})")
print(f"  Gap:                {arith_mean_daily - geo_mean_daily:>10.6f}")
print(f"  Gap ≈ Var/2:        {gap_approx:>10.6f}  (approximation)")

print(f"\n{'─'*58}")
print("  ANNUALISED")
print(f"{'─'*58}")
print(f"  Arithmetic Mean:    {arith_mean_ann:>10.4%}")
print(f"  Geometric Mean:     {geo_mean_ann:>10.4%}")
print(f"  Gap:                {arith_mean_ann - geo_mean_ann:>10.4%}")

print(f"\n{'─'*58}")
print("  VERIFICATION")
print(f"{'─'*58}")
print(f"  Full Period HPR:    {hpr_full:>10.4%}")
print(f"  (1+geo)^T - 1:     {verify:>10.4%}  ✓ matches")

print(f"\n{'─'*58}")
print("  INTERPRETATION")
print(f"{'─'*58}")
print(f"  Arithmetic mean overstates compound growth by {arith_mean_ann - geo_mean_ann:.2%} per year.")
print(f"  A manager reporting {arith_mean_ann:.2%} arithmetic vs")
print(f"  {geo_mean_ann:.2%} geometric — same portfolio, different story.")
print(f"{'='*58}")

ARITHMETIC vs GEOMETRIC MEAN — SPY Daily Returns (2019–2023)

──────────────────────────────────────────────────────────
  DAILY
──────────────────────────────────────────────────────────
  Arithmetic Mean:      0.000599  (0.0599%)
  Geometric Mean:       0.000511  (0.0511%)
  Gap:                  0.000089
  Gap ≈ Var/2:          0.000088  (approximation)

──────────────────────────────────────────────────────────
  ANNUALISED
──────────────────────────────────────────────────────────
  Arithmetic Mean:      15.1002%
  Geometric Mean:       13.7308%
  Gap:                   1.3694%

──────────────────────────────────────────────────────────
  VERIFICATION
──────────────────────────────────────────────────────────
  Full Period HPR:      89.9872%
  (1+geo)^T - 1:       89.9872%  ✓ matches

──────────────────────────────────────────────────────────
  INTERPRETATION
──────────────────────────────────────────────────────────
  Arithmetic mean overstates compound growth by 1.37% per year.


## 3. Money-Weighted Return (MWRR)

The **Money-Weighted Return** is the Internal Rate of Return (IRR)
of all cash flows into and out of a portfolio.

It answers: *what return did this specific investor earn,
given when they put money in and took money out?*

$$0 = CF_0 + \frac{CF_1}{(1+r)^1} + \frac{CF_2}{(1+r)^2} + \cdots + \frac{CF_T}{(1+r)^T}$$

Where $r$ is the MWRR — solved numerically.

**Key property:** MWRR is sensitive to the timing and size of cash flows.

If an investor adds money just before a bad period
or withdraws just before a good period,
their MWRR will be lower than the portfolio's actual performance.

This is why MWRR reflects the **investor's experience**,
not the **manager's skill**.

We simulate a realistic investment scenario:
an investor who buys SPY in tranches over 5 years.

In [4]:
# =============================================================================
# CELL 9 — Money-Weighted Return (MWRR)
# =============================================================================
# We simulate an investor who buys SPY in three tranches:
#   - Jan 2019: invests $10,000 (initial purchase)
#   - Jan 2021: invests $5,000  (adds at market high pre-COVID recovery)
#   - Jan 2023: invests $5,000  (adds after 2022 drawdown)
#   - Dec 2023: sells entire position (ending value)
#
# MWRR = IRR of these cash flows
# Outflows (investments) = negative
# Inflows (proceeds)     = positive
# =============================================================================

# ── GET PRICES AT EACH DATE ───────────────────────────────────────────────────
dates = {
    "2019-01-02": {"shares": 0,   "invest": -10000},
    "2021-01-04": {"shares": 0,   "invest": -5000},
    "2023-01-03": {"shares": 0,   "invest": -5000},
    "2023-12-29": {"shares": 0,   "invest": 0},      # exit
}

# Get actual prices on those dates
for date_str in dates:
    date = pd.Timestamp(date_str)
    # Find closest available price
    closest = spy.index[spy.index.get_indexer([date], method='nearest')[0]]
    dates[date_str]["price"] = spy[closest]

# ── CALCULATE SHARES PURCHASED ────────────────────────────────────────────────
total_shares = 0
cash_flows   = []
cf_dates     = []

for date_str, info in dates.items():
    price = info["price"]
    invest = info["invest"]

    if invest < 0:  # buying
        shares_bought = abs(invest) / price
        total_shares += shares_bought
        cash_flows.append(invest)  # negative = outflow
    else:  # exit — sell all shares
        exit_value = total_shares * price
        cash_flows.append(exit_value)  # positive = inflow

    cf_dates.append(pd.Timestamp(date_str))

# ── SOLVE FOR MWRR (IRR) ─────────────────────────────────────────────────────
# Convert to time-weighted periods (years from start)
t0 = cf_dates[0]
periods_years = [(d - t0).days / 365.25 for d in cf_dates]

def npv(rate, cash_flows, periods):
    return sum(cf / (1 + rate) ** t for cf, t in zip(cash_flows, periods))

# Solve NPV = 0 for rate
mwrr = optimize.brentq(npv, -0.5, 10.0,
                        args=(cash_flows, periods_years))

# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
total_invested  = sum(abs(cf) for cf in cash_flows if cf < 0)
exit_value      = cash_flows[-1]
total_profit    = exit_value - total_invested

print(f"{'='*58}")
print("MONEY-WEIGHTED RETURN (MWRR) — SPY Investment Simulation")
print(f"{'='*58}")

print(f"\n  CASH FLOWS")
print(f"  {'Date':<14} {'Price':>8} {'Cash Flow':>12} {'Shares':>10}")
print(f"  {'-'*46}")

share_tracker = 0
for i, (date_str, info) in enumerate(dates.items()):
    price  = info["price"]
    cf     = cash_flows[i]
    if cf < 0:
        shares = abs(cf) / price
        share_tracker += shares
        print(f"  {date_str:<14} ${price:>7.2f} ${cf:>11,.0f} {shares:>10.2f}")
    else:
        print(f"  {date_str:<14} ${price:>7.2f} ${cf:>11,.0f} {share_tracker:>10.2f} (exit)")

print(f"\n  SUMMARY")
print(f"  Total Invested:    ${total_invested:>10,.0f}")
print(f"  Exit Value:        ${exit_value:>10,.0f}")
print(f"  Total Profit:      ${total_profit:>10,.0f}")
print(f"\n  MWRR (annualised): {mwrr:.4%}")

print(f"\n  INTERPRETATION")
print(f"  The investor's actual annualised return was {mwrr:.2%}.")
print(f"  Compare to SPY geometric mean return: {geo_mean_ann:.2%}")
print(f"  Difference: {mwrr - geo_mean_ann:.2%}")
if mwrr < geo_mean_ann:
    print(f"  The investor underperformed the fund due to cash flow timing.")
else:
    print(f"  The investor outperformed the fund due to cash flow timing.")
print(f"{'='*58}")

MONEY-WEIGHTED RETURN (MWRR) — SPY Investment Simulation

  CASH FLOWS
  Date              Price    Cash Flow     Shares
  ----------------------------------------------
  2019-01-02     $ 250.18 $    -10,000      39.97
  2021-01-04     $ 368.79 $     -5,000      13.56
  2023-01-03     $ 380.82 $     -5,000      13.13
  2023-12-29     $ 475.31 $     31,684      66.66 (exit)

  SUMMARY
  Total Invested:    $    20,000
  Exit Value:        $    31,684
  Total Profit:      $    11,684

  MWRR (annualised): 13.4225%

  INTERPRETATION
  The investor's actual annualised return was 13.42%.
  Compare to SPY geometric mean return: 13.73%
  Difference: -0.31%
  The investor underperformed the fund due to cash flow timing.


## 4. Time-Weighted Return (TWRR)

The **Time-Weighted Return** eliminates the effect of cash flow timing.

It measures the compound growth of **one unit of money** invested
at the start — regardless of when investors add or withdraw funds.

$$TWRR = \left(\prod_{t=1}^{T} (1 + HPR_t)\right)^{1/T} - 1$$

Where each $HPR_t$ is the sub-period return between cash flows.

**The key difference from MWRR:**

| | MWRR | TWRR |
|--|------|------|
| Measures | Investor return | Manager return |
| Affected by cash flow timing | Yes | No |
| CFA standard for manager performance | No | Yes |
| Required by GIPS | No | Yes |

TWRR is the **Global Investment Performance Standards (GIPS)** standard
for reporting manager performance — because it is not distorted
by cash flows the manager does not control.

In [5]:
# =============================================================================
# CELL 11 — Time-Weighted Return (TWRR)
# =============================================================================
# We use the same dates as the MWRR simulation.
# TWRR chain-links the sub-period HPRs between each cash flow date.
#
# Sub-periods:
#   Period 1: 2019-01-02 → 2021-01-04
#   Period 2: 2021-01-04 → 2023-01-03
#   Period 3: 2023-01-03 → 2023-12-29
# =============================================================================

# ── SUB-PERIOD PRICES ─────────────────────────────────────────────────────────
sub_period_dates = ["2019-01-02", "2021-01-04", "2023-01-03", "2023-12-29"]
sub_period_prices = []

for date_str in sub_period_dates:
    date    = pd.Timestamp(date_str)
    closest = spy.index[spy.index.get_indexer([date], method='nearest')[0]]
    sub_period_prices.append((date_str, spy[closest]))

# ── COMPUTE SUB-PERIOD HPRs ───────────────────────────────────────────────────
sub_period_hprs = []
for i in range(len(sub_period_prices) - 1):
    start_date, start_price = sub_period_prices[i]
    end_date,   end_price   = sub_period_prices[i + 1]
    hpr = (end_price / start_price) - 1
    days = (pd.Timestamp(end_date) - pd.Timestamp(start_date)).days
    sub_period_hprs.append({
        "start": start_date,
        "end":   end_date,
        "start_price": start_price,
        "end_price":   end_price,
        "hpr":   hpr,
        "days":  days
    })

# ── CHAIN-LINK TO GET TWRR ────────────────────────────────────────────────────
total_days   = sum(p["days"] for p in sub_period_hprs)
total_years  = total_days / 365.25

# Chain-link product
chain_product = np.prod([1 + p["hpr"] for p in sub_period_hprs])

# Annualise
twrr_ann = chain_product ** (1 / total_years) - 1

# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
print(f"{'='*62}")
print("TIME-WEIGHTED RETURN (TWRR) — SPY Investment Simulation")
print(f"{'='*62}")

print(f"\n  SUB-PERIOD RETURNS")
print(f"  {'Period':<28} {'Start':>7} {'End':>7} {'HPR':>9} {'Days':>6}")
print(f"  {'-'*58}")

for p in sub_period_hprs:
    period_label = f"{p['start']} → {p['end']}"
    print(f"  {period_label:<28} ${p['start_price']:>6.2f} ${p['end_price']:>6.2f} "
          f"{p['hpr']:>8.2%} {p['days']:>6}")

print(f"\n  Chain product: {chain_product:.6f}")
print(f"  Total period:  {total_years:.2f} years")

print(f"\n  {'='*40}")
print(f"  TWRR (annualised):  {twrr_ann:.4%}")
print(f"  MWRR (annualised):  {mwrr:.4%}")
print(f"  Difference:         {twrr_ann - mwrr:.4%}")
print(f"  {'='*40}")

print(f"""
  INTERPRETATION
  TWRR = {twrr_ann:.2%} — this is the manager's performance.
  MWRR = {mwrr:.2%} — this is the investor's actual experience.

  The {abs(twrr_ann - mwrr):.2%} gap exists because the investor added capital
  at prices that were {'above' if mwrr < twrr_ann else 'below'} the fund's average growth trajectory.

  CFA Rule: Use TWRR to evaluate the manager.
             Use MWRR to evaluate the investor's decisions.
""")
print(f"{'='*62}")

TIME-WEIGHTED RETURN (TWRR) — SPY Investment Simulation

  SUB-PERIOD RETURNS
  Period                         Start     End       HPR   Days
  ----------------------------------------------------------
  2019-01-02 → 2021-01-04      $250.18 $368.79   47.41%    733
  2021-01-04 → 2023-01-03      $368.79 $380.82    3.26%    729
  2023-01-03 → 2023-12-29      $380.82 $475.31   24.81%    360

  Chain product: 1.899872
  Total period:  4.99 years

  TWRR (annualised):  13.7300%
  MWRR (annualised):  13.4225%
  Difference:         0.3075%

  INTERPRETATION
  TWRR = 13.73% — this is the manager's performance.
  MWRR = 13.42% — this is the investor's actual experience.

  The 0.31% gap exists because the investor added capital
  at prices that were above the fund's average growth trajectory.

  CFA Rule: Use TWRR to evaluate the manager.
             Use MWRR to evaluate the investor's decisions.



## 5. Annualising Returns

A return means nothing without knowing the time period it covers.

Annualising allows us to compare returns across different horizons
on a common basis — the standard in the CFA curriculum.

**The general formula:**

$$r_{annual} = (1 + r_{period})^{m} - 1$$

Where $m$ is the number of periods per year:

| Period | m |
|--------|---|
| Daily | 252 (trading days) |
| Weekly | 52 |
| Monthly | 12 |
| Quarterly | 4 |
| Semi-annual | 2 |

**Important:** this assumes returns compound at the same rate
each period — which is an assumption, not a guarantee.

We also cover **continuous compounding** — the limiting case
used in options pricing and fixed income:

$$r_{continuous} = e^{r_{annual}} - 1 \quad \Leftrightarrow \quad r_{annual} = \ln(1 + r_{continuous})$$

In [7]:
# =============================================================================
# CELL 13 — Annualising Returns
# =============================================================================

# ── USE SPY RETURNS ACROSS DIFFERENT FREQUENCIES ─────────────────────────────

# Daily return (geometric mean)
r_daily     = geo_mean_daily

# Weekly — resample to week end
spy_weekly  = spy.resample("W").last()
hpr_weekly  = (spy_weekly / spy_weekly.shift(1) - 1).dropna()
r_weekly    = (np.prod(1 + hpr_weekly) ** (1 / len(hpr_weekly))) - 1

# Monthly
spy_monthly = spy.resample("ME").last()
hpr_monthly = (spy_monthly / spy_monthly.shift(1) - 1).dropna()
r_monthly   = (np.prod(1 + hpr_monthly) ** (1 / len(hpr_monthly))) - 1

# Quarterly
spy_quarterly = spy.resample("QE").last()
hpr_quarterly = (spy_quarterly / spy_quarterly.shift(1) - 1).dropna()
r_quarterly   = (np.prod(1 + hpr_quarterly) ** (1 / len(hpr_quarterly))) - 1

# Full period — already computed
r_full_annual = twrr_ann

# ── ANNUALISE EACH ────────────────────────────────────────────────────────────
periods_per_year = {"Daily": 252, "Weekly": 52, "Monthly": 12, "Quarterly": 4}
raw_returns      = {"Daily": r_daily, "Weekly": r_weekly,
                    "Monthly": r_monthly, "Quarterly": r_quarterly}

print(f"{'='*65}")
print("ANNUALISED RETURNS — SPY (2019–2023)")
print(f"{'='*65}")
print(f"\n  {'Frequency':<12} {'Period Return':>15} {'Ann. Return':>13} {'Formula'}")
print(f"  {'-'*60}")

ann_returns = {}
for freq, r_period in raw_returns.items():
    m          = periods_per_year[freq]
    r_ann      = (1 + r_period) ** m - 1
    ann_returns[freq] = r_ann
    print(f"  {freq:<12} {r_period:>14.6f} {r_ann:>12.4%}   (1+r)^{m}-1")

print(f"\n  Note: small differences across frequencies reflect")
print(f"  sampling variation, not calculation errors.")
print(f"  Daily TWRR ({ann_returns['Daily']:.2%}) is the most precise estimate.")

# ── CONTINUOUS COMPOUNDING ────────────────────────────────────────────────────
print(f"\n{'─'*65}")
print("  CONTINUOUS COMPOUNDING")
print(f"{'─'*65}")

# Convert annual discrete return to continuous
r_continuous = np.log(1 + twrr_ann)
r_back       = np.exp(r_continuous) - 1

print(f"\n  Discrete annual return:     {twrr_ann:.6f}  ({twrr_ann:.4%})")
print(f"  Continuous equivalent:      {r_continuous:.6f}  ({r_continuous:.4%})")
print(f"  Back to discrete:           {r_back:.6f}  ({r_back:.4%})  ✓")

print(f"""
  Continuous compounding assumes returns compound instantaneously.
  It is used in Black-Scholes and fixed income duration models.
  For the CFA exam: ln(1 + r_discrete) = r_continuous
""")
print(f"{'='*65}")

ANNUALISED RETURNS — SPY (2019–2023)

  Frequency      Period Return   Ann. Return Formula
  ------------------------------------------------------------
  Daily              0.000511     13.7308%   (1+r)^252-1
  Weekly             0.002438     13.4961%   (1+r)^52-1
  Monthly            0.009636     12.1962%   (1+r)^12-1
  Quarterly          0.027766     11.5775%   (1+r)^4-1

  Note: small differences across frequencies reflect
  sampling variation, not calculation errors.
  Daily TWRR (13.73%) is the most precise estimate.

─────────────────────────────────────────────────────────────────
  CONTINUOUS COMPOUNDING
─────────────────────────────────────────────────────────────────

  Discrete annual return:     0.137300  (13.7300%)
  Continuous equivalent:      0.128657  (12.8657%)
  Back to discrete:           0.137300  (13.7300%)  ✓

  Continuous compounding assumes returns compound instantaneously.
  It is used in Black-Scholes and fixed income duration models.
  For the CFA exam: ln(

## 6. Real vs Nominal Returns

Everything we have computed so far is a **nominal return** —
it does not account for inflation.

The **real return** adjusts for the loss of purchasing power:

$$1 + r_{real} = \frac{1 + r_{nominal}}{1 + r_{inflation}}$$

Which simplifies to the **Fisher equation approximation**:

$$r_{real} \approx r_{nominal} - r_{inflation}$$

The approximation works well for low inflation.
It breaks down when inflation is high — as it was in 2021-2022.

We pull **real CPI data from FRED** to compute the actual
inflation rate over our sample period and adjust SPY's return.

FRED series: **CPIAUCSL** — Consumer Price Index for All Urban Consumers

In [8]:
# =============================================================================
# CELL 15 — Real vs Nominal Returns using FRED CPI Data
# =============================================================================

def get_fred_series(series_id, api_key, from_date="2019-01-01", to_date="2024-01-01"):
    """Pull a data series from FRED API."""
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id":         series_id,
        "api_key":           api_key,
        "file_type":         "json",
        "observation_start": from_date,
        "observation_end":   to_date,
    }
    response = requests.get(url, params=params)
    data     = response.json()

    df = pd.DataFrame(data["observations"])
    df["date"]  = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.set_index("date").dropna()
    return df["value"]

# Pull CPI data
print("Pulling CPI data from FRED...")
cpi = get_fred_series("CPIAUCSL", FRED_KEY)
print(f"  ✓ CPI — {len(cpi)} monthly observations loaded")
print(f"  Period: {cpi.index[0].date()} → {cpi.index[-1].date()}")

# ── COMPUTE INFLATION RATE ────────────────────────────────────────────────────
# Monthly CPI inflation
cpi_monthly_inf = (cpi / cpi.shift(1) - 1).dropna()

# Annual inflation — chain-link monthly
cpi_start = cpi.iloc[0]
cpi_end   = cpi.iloc[-1]
n_years   = (cpi.index[-1] - cpi.index[0]).days / 365.25
inflation_ann = (cpi_end / cpi_start) ** (1 / n_years) - 1

# ── REAL RETURN ───────────────────────────────────────────────────────────────
r_nominal = twrr_ann
r_real_exact  = (1 + r_nominal) / (1 + inflation_ann) - 1
r_real_approx = r_nominal - inflation_ann

# ── YEAR BY YEAR BREAKDOWN ────────────────────────────────────────────────────
spy_annual = spy.resample("YE").last()
hpr_annual = (spy_annual / spy_annual.shift(1) - 1).dropna()

cpi_annual     = cpi.resample("YE").last()
inf_annual     = (cpi_annual / cpi_annual.shift(1) - 1).dropna()

# Align
common_years = hpr_annual.index.intersection(inf_annual.index)
hpr_y = hpr_annual[common_years]
inf_y = inf_annual[common_years]
real_y = (1 + hpr_y) / (1 + inf_y) - 1

# ── PRINT RESULTS ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("REAL vs NOMINAL RETURNS — SPY (2019–2023)")
print(f"{'='*60}")

print(f"\n  CPI (CPIAUCSL): {cpi_start:.2f} → {cpi_end:.2f}")
print(f"  Annualised Inflation: {inflation_ann:.4%}")

print(f"\n  {'─'*55}")
print(f"  Nominal Return (TWRR):        {r_nominal:.4%}")
print(f"  Inflation Rate:               {inflation_ann:.4%}")
print(f"  Real Return (exact):          {r_real_exact:.4%}")
print(f"  Real Return (approximation):  {r_real_approx:.4%}")
print(f"  Approximation error:          {r_real_exact - r_real_approx:.4%}")

print(f"\n  {'─'*55}")
print(f"  YEAR-BY-YEAR BREAKDOWN")
print(f"  {'─'*55}")
print(f"  {'Year':<8} {'Nominal':>10} {'Inflation':>11} {'Real':>10}")
print(f"  {'-'*42}")
for year in common_years:
    print(f"  {year.year:<8} {hpr_y[year]:>10.2%} {inf_y[year]:>10.2%} {real_y[year]:>10.2%}")

print(f"\n  {'─'*55}")
print(f"  SPY earned {r_nominal:.2%} nominally but only")
print(f"  {r_real_exact:.2%} in real purchasing power terms.")
print(f"  Inflation consumed {inflation_ann:.2%} of returns per year.")
print(f"{'='*60}")

Pulling CPI data from FRED...
  ✓ CPI — 61 monthly observations loaded
  Period: 2019-01-01 → 2024-01-01

REAL vs NOMINAL RETURNS — SPY (2019–2023)

  CPI (CPIAUCSL): 252.56 → 309.70
  Annualised Inflation: 4.1638%

  ───────────────────────────────────────────────────────
  Nominal Return (TWRR):        13.7300%
  Inflation Rate:               4.1638%
  Real Return (exact):          9.1838%
  Real Return (approximation):  9.5662%
  Approximation error:          -0.3824%

  ───────────────────────────────────────────────────────
  YEAR-BY-YEAR BREAKDOWN
  ───────────────────────────────────────────────────────
  Year        Nominal   Inflation       Real
  ------------------------------------------
  2020         16.16%      1.32%     14.65%
  2021         27.04%      7.17%     18.53%
  2022        -19.48%      6.40%    -24.33%
  2023         24.29%      3.32%     20.30%

  ───────────────────────────────────────────────────────
  SPY earned 13.73% nominally but only
  9.18% in real pu

## 7. Leveraged Returns

Leverage amplifies both gains and losses.

When an investor borrows money to invest, their return on equity becomes:

$$r_{leveraged} = r_{portfolio} + \frac{D}{E}(r_{portfolio} - r_{borrowing})$$

Where:
- $r_{portfolio}$ = return on total assets
- $D/E$ = debt-to-equity ratio (leverage ratio)
- $r_{borrowing}$ = cost of borrowed funds

If $r_{portfolio} > r_{borrowing}$ → leverage amplifies gains

If $r_{portfolio} < r_{borrowing}$ → leverage amplifies losses

**This is the most dangerous concept in this notebook.**

A 2x leveraged position in SPY during 2022
would have turned a −19.48% loss into approximately −42%
before accounting for borrowing costs.

We simulate leveraged SPY positions at different leverage ratios
using realistic margin borrowing rates.

In [9]:
# =============================================================================
# CELL 17 — Leveraged Returns
# =============================================================================
# We simulate an investor with $10,000 equity who borrows additional
# capital to invest in SPY at different leverage ratios.
#
# Borrowing rate: we use the Federal Funds Rate as a proxy
# pulled from FRED (series: FEDFUNDS)
# =============================================================================

# Pull Fed Funds Rate from FRED
print("Pulling Fed Funds Rate from FRED...")
fed_funds = get_fred_series("FEDFUNDS", FRED_KEY)
fed_funds_ann = fed_funds / 100  # convert from % to decimal
print(f"  ✓ FEDFUNDS — {len(fed_funds)} monthly observations loaded")

# Annual average borrowing rate over the period
r_borrow_ann = fed_funds_ann.mean()
print(f"  Average Fed Funds Rate (2019–2023): {r_borrow_ann:.4%}")

# ── SIMULATE LEVERAGED RETURNS ────────────────────────────────────────────────
equity        = 10_000
leverage_ratios = [1.0, 1.5, 2.0, 3.0]

# Use annual SPY returns for clarity
print(f"\n{'='*68}")
print("LEVERAGED RETURNS — SPY Annual Returns at Different Leverage Ratios")
print(f"{'='*68}")

print(f"\n  Equity: ${equity:,} | Borrowing Rate: {r_borrow_ann:.2%} (avg Fed Funds)")

print(f"\n  {'Year':<6} {'SPY Return':>11} ", end="")
for lr in leverage_ratios:
    print(f"  {'L='+str(lr)+'x':>8}", end="")
print()
print(f"  {'-'*62}")

yearly_leveraged = {lr: [] for lr in leverage_ratios}

for year in common_years:
    r_port = hpr_y[year]
    print(f"  {year.year:<6} {r_port:>10.2%} ", end="")
    for lr in leverage_ratios:
        debt_to_equity = lr - 1
        r_lev = r_port + debt_to_equity * (r_port - r_borrow_ann)
        yearly_leveraged[lr].append(r_lev)
        print(f"  {r_lev:>8.2%}", end="")
    print()

# ── COMPOUND OVER FULL PERIOD ─────────────────────────────────────────────────
print(f"\n  {'-'*62}")
print(f"  {'Compounded':>16} ", end="")
for lr in leverage_ratios:
    r_compound = np.prod([1 + r for r in yearly_leveraged[lr]]) - 1
    print(f"  {r_compound:>8.2%}", end="")
print()

print(f"\n  {'Final $10k':>16} ", end="")
for lr in leverage_ratios:
    r_compound = np.prod([1 + r for r in yearly_leveraged[lr]]) - 1
    final_value = equity * (1 + r_compound)
    print(f"  ${final_value:>7,.0f}", end="")
print()

print(f"\n{'='*68}")
print(f"""
  KEY OBSERVATIONS

  1x (no leverage):  baseline SPY return, $10k grows normally
  1.5x leverage:     amplifies gains in good years, losses in bad
  2x leverage:       2022 loss becomes catastrophic (~-42%)
  3x leverage:       a single bad year can wipe out years of gains

  Leverage ratio of {max(leverage_ratios)}x in 2022:
  SPY lost {hpr_y[common_years[-2]]:.2%} → leveraged investor lost
  approximately {hpr_y[common_years[-2]] + (max(leverage_ratios)-1)*(hpr_y[common_years[-2]] - r_borrow_ann):.2%}

  CFA Rule: Leverage magnifies BOTH returns AND risk.
  Always assess the downside scenario before applying leverage.
""")
print(f"{'='*68}")

Pulling Fed Funds Rate from FRED...
  ✓ FEDFUNDS — 61 monthly observations loaded
  Average Fed Funds Rate (2019–2023): 1.9211%

LEVERAGED RETURNS — SPY Annual Returns at Different Leverage Ratios

  Equity: $10,000 | Borrowing Rate: 1.92% (avg Fed Funds)

  Year    SPY Return     L=1.0x    L=1.5x    L=2.0x    L=3.0x
  --------------------------------------------------------------
  2020       16.16%     16.16%    23.28%    30.40%    44.64%
  2021       27.04%     27.04%    39.59%    52.15%    77.26%
  2022      -19.48%    -19.48%   -30.18%   -40.88%   -62.29%
  2023       24.29%     24.29%    35.47%    46.65%    69.02%

  --------------------------------------------------------------
        Compounded     47.68%    62.77%    72.01%    63.43%

        Final $10k   $ 14,768  $ 16,277  $ 17,201  $ 16,343


  KEY OBSERVATIONS

  1x (no leverage):  baseline SPY return, $10k grows normally
  1.5x leverage:     amplifies gains in good years, losses in bad
  2x leverage:       2022 loss beco

## 8. CFA Exam Style Practice Problems

Three problems written in CFA exam style.

Try each one before reading the solution.

In [10]:
# =============================================================================
# CELL 19 — CFA Exam Style Practice Problems
# =============================================================================

print("=" * 65)
print("PRACTICE PROBLEMS — Rates and Returns")
print("=" * 65)

# ── PROBLEM 1 ─────────────────────────────────────────────────────────────────
print("""
PROBLEM 1
─────────────────────────────────────────────────────────────
An investor buys 100 shares of a stock at $45.00 per share.
After 18 months the stock is trading at $52.50 and has paid
$1.20 per share in dividends during the holding period.

A) Calculate the holding period return.
B) Annualise the holding period return.
─────────────────────────────────────────────────────────────""")

P0       = 45.00
Pt       = 52.50
D        = 1.20
months   = 18

hpr_p1   = (Pt - P0 + D) / P0
r_ann_p1 = (1 + hpr_p1) ** (12 / months) - 1

print(f"ANSWER:")
print(f"  A) HPR = (52.50 - 45.00 + 1.20) / 45.00 = {hpr_p1:.4%}")
print(f"  B) Annualised = (1 + {hpr_p1:.4f})^(12/18) - 1 = {r_ann_p1:.4%}")

# ── PROBLEM 2 ─────────────────────────────────────────────────────────────────
print(f"""
{'='*65}
PROBLEM 2
─────────────────────────────────────────────────────────────
A portfolio manager reports the following annual returns:

  Year 1:  +18.0%
  Year 2:  +12.0%
  Year 3:  -22.0%
  Year 4:  +25.0%

A) Calculate the arithmetic mean return.
B) Calculate the geometric mean return.
C) Which is more appropriate for evaluating the manager's
   compound performance? Why?
─────────────────────────────────────────────────────────────""")

returns_p2 = [0.18, 0.12, -0.22, 0.25]

arith_p2 = np.mean(returns_p2)
geo_p2   = np.prod([1 + r for r in returns_p2]) ** (1 / len(returns_p2)) - 1

print(f"ANSWER:")
print(f"  A) Arithmetic = ({'+'.join([f'{r:.0%}' for r in returns_p2])}) / 4")
print(f"               = {arith_p2:.4%}")
print(f"  B) Geometric  = (1.18 × 1.12 × 0.78 × 1.25)^(1/4) - 1")
print(f"               = {geo_p2:.4%}")
print(f"  C) Geometric mean — it reflects actual compound growth.")
print(f"     A $1 investment grows to ${np.prod([1+r for r in returns_p2]):.4f}")
print(f"     over 4 years, consistent with the geometric mean.")
print(f"     The arithmetic mean ({arith_p2:.2%}) overstates performance.")

# ── PROBLEM 3 ─────────────────────────────────────────────────────────────────
print(f"""
{'='*65}
PROBLEM 3
─────────────────────────────────────────────────────────────
A fund reports a TWRR of 14.50% and an MWRR of 11.20%
over the same period.

A) What does the gap between TWRR and MWRR tell you?
B) Which return should be used to evaluate the manager?
C) An investor added a large sum just before a period of
   poor performance. How does this affect MWRR vs TWRR?
─────────────────────────────────────────────────────────────""")

twrr_p3 = 0.1450
mwrr_p3 = 0.1120

print(f"ANSWER:")
print(f"  A) TWRR ({twrr_p3:.2%}) > MWRR ({mwrr_p3:.2%}) by {twrr_p3-mwrr_p3:.2%}.")
print(f"     The investor's cash flow timing hurt their actual returns.")
print(f"     The fund performed well but the investor added capital")
print(f"     at an unfavourable time.")
print(f"  B) TWRR — it measures the manager's performance independently")
print(f"     of investor cash flow decisions. Required by GIPS.")
print(f"  C) Adding capital before poor performance increases the")
print(f"     weight of that poor period in MWRR, dragging it below TWRR.")
print(f"     TWRR is unaffected — it treats each sub-period equally.")
print(f"{'='*65}")

PRACTICE PROBLEMS — Rates and Returns

PROBLEM 1
─────────────────────────────────────────────────────────────
An investor buys 100 shares of a stock at $45.00 per share.
After 18 months the stock is trading at $52.50 and has paid
$1.20 per share in dividends during the holding period.

A) Calculate the holding period return.
B) Annualise the holding period return.
─────────────────────────────────────────────────────────────
ANSWER:
  A) HPR = (52.50 - 45.00 + 1.20) / 45.00 = 19.3333%
  B) Annualised = (1 + 0.1933)^(12/18) - 1 = 12.5057%

PROBLEM 2
─────────────────────────────────────────────────────────────
A portfolio manager reports the following annual returns:

  Year 1:  +18.0%
  Year 2:  +12.0%
  Year 3:  -22.0%
  Year 4:  +25.0%

A) Calculate the arithmetic mean return.
B) Calculate the geometric mean return.
C) Which is more appropriate for evaluating the manager's
   compound performance? Why?
─────────────────────────────────────────────────────────────
ANSWER:
  A) Arithm

## 9. Key Takeaways

---

### Return measures and when to use them

| Measure | Use Case | Affected by Cash Flows |
|---------|----------|----------------------|
| HPR | Single period, any horizon | No |
| Arithmetic Mean | Forward-looking, expected return | No |
| Geometric Mean | Compound growth, historical performance | No |
| MWRR | Investor's actual experience | Yes |
| TWRR | Manager performance evaluation (GIPS) | No |

---

### The three rules to remember

**1. Geometric mean ≤ Arithmetic mean — always.**
The gap equals approximately variance/2.
Use geometric for performance reporting, arithmetic for forecasting.

**2. TWRR for managers, MWRR for investors.**
TWRR eliminates the distortion of cash flow timing.
MWRR captures the investor's actual dollar-weighted experience.

**3. Always adjust for inflation when assessing purchasing power.**
SPY returned 13.73% nominally but only 9.18% in real terms.
Inflation consumed 4.16% per year — that is not trivial.

---

### What comes next

| Notebook | Topic |
|----------|-------|
| ✅ QM 01 | Rates and Returns |
| ✅ QM 02 | Time Value of Money |
| ✅ QM 03 | Statistical Measures of Asset Returns |
| 🔜 QM 04 | Probability Concepts |
| 🔜 QM 05 | Common Probability Distributions |
| 🔜 QM 06 | Sampling & Estimation |
| 🔜 QM 07 | Hypothesis Testing |
| 🔜 QM 08 | Introduction to Linear Regression |

---

*All notebooks are free. Always.*